# NB11 - Data Contracts & CI Validation
## Background
A data contract is a schema agreement between data producers and consumers that specifies: column types, allowed value ranges, missing value rates, expected distributions, and freshness requirements. When a dataset violates its contract — a column fills with NaN, a distribution drifts, or an undocumented schema change occurs — downstream models break silently.

Great Expectations, Pandera, and dbt tests are the production tools for data contract enforcement. Statistical tests (KS test, chi-squared, Jensen-Shannon divergence) detect distribution drift beyond schema-level checks. Embedding these checks in CI/CD pipelines catches data quality regressions before they reach production.

## What You'll Learn
- Defining a data contract: schema, ranges, distributions
- Statistical validation: KS test for distribution drift
- CI gate implementation: pass/fail with severity levels
- Contract violation reporting with actionable messages

## Why This Matters
Silent data quality regressions are one of the most common causes of model performance degradation in production. A data contract catches these issues at data ingestion time, not weeks later when model metrics degrade.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Tuple
from scipy import stats

# ── Data Contract Definition ───────────────────────────────────────────────
@dataclass
class ColumnContract:
    name: str
    dtype: str  # 'numeric', 'categorical', 'bool'
    min_val: Optional[float] = None
    max_val: Optional[float] = None
    allowed_values: Optional[List] = None
    max_null_rate: float = 0.05
    expected_mean: Optional[float] = None
    expected_std: Optional[float] = None
    mean_tolerance: float = 0.2  # fraction
    std_tolerance: float = 0.3

@dataclass
class DataContract:
    name: str
    version: str
    columns: List[ColumnContract] = field(default_factory=list)
    expected_n_rows: Optional[int] = None
    row_tolerance: float = 0.1  # allowed fraction deviation

    def add_column(self, col: ColumnContract):
        self.columns.append(col)

@dataclass
class ValidationResult:
    check_name: str
    column: str
    passed: bool
    severity: str  # 'error', 'warning'
    message: str

# ── Contract validator ─────────────────────────────────────────────────────
class DataContractValidator:
    def __init__(self, contract: DataContract):
        self.contract = contract
        self.results: List[ValidationResult] = []

    def validate(self, data: Dict[str, np.ndarray]) -> bool:
        self.results = []

        # Row count check
        n_rows = len(next(iter(data.values())))
        if self.contract.expected_n_rows:
            expected = self.contract.expected_n_rows
            deviation = abs(n_rows - expected) / expected
            self.results.append(ValidationResult(
                'row_count', 'global',
                deviation <= self.contract.row_tolerance,
                'error' if deviation > 0.2 else 'warning',
                f'Row count: {n_rows} (expected {expected}, deviation={deviation:.1%})'
            ))

        for col_contract in self.contract.columns:
            col = col_contract.name
            if col not in data:
                self.results.append(ValidationResult(
                    'column_exists', col, False, 'error',
                    f'Column {col} missing from dataset'))
                continue
            values = data[col]

            # Null rate
            null_rate = np.isnan(values).mean() if values.dtype == float else 0.0
            self.results.append(ValidationResult(
                'null_rate', col,
                null_rate <= col_contract.max_null_rate,
                'error',
                f'Null rate: {null_rate:.3f} (max allowed: {col_contract.max_null_rate})'))

            if col_contract.dtype == 'numeric':
                non_null = values[~np.isnan(values)] if values.dtype == float else values

                # Range check
                if col_contract.min_val is not None:
                    in_range = (non_null >= col_contract.min_val).all()
                    self.results.append(ValidationResult(
                        'range_min', col, bool(in_range), 'error',
                        f'Min value: {non_null.min():.3f} (expected >= {col_contract.min_val})'))
                if col_contract.max_val is not None:
                    in_range = (non_null <= col_contract.max_val).all()
                    self.results.append(ValidationResult(
                        'range_max', col, bool(in_range), 'error',
                        f'Max value: {non_null.max():.3f} (expected <= {col_contract.max_val})'))

                # Distribution drift
                if col_contract.expected_mean is not None:
                    actual_mean = non_null.mean()
                    mean_dev = abs(actual_mean - col_contract.expected_mean) / \
                               (abs(col_contract.expected_mean) + 1e-10)
                    self.results.append(ValidationResult(
                        'mean_drift', col,
                        mean_dev <= col_contract.mean_tolerance,
                        'warning',
                        f'Mean: {actual_mean:.3f} (expected {col_contract.expected_mean:.3f}, '
                        f'drift={mean_dev:.1%})'))

            elif col_contract.dtype == 'categorical':
                if col_contract.allowed_values:
                    unknown = set(values) - set(col_contract.allowed_values)
                    self.results.append(ValidationResult(
                        'allowed_values', col, len(unknown) == 0, 'error',
                        f'Unknown categories: {unknown}' if unknown else 'OK'))

        n_errors = sum(1 for r in self.results if not r.passed and r.severity == 'error')
        n_warnings = sum(1 for r in self.results if not r.passed and r.severity == 'warning')
        return n_errors == 0

    def report(self):
        n_pass = sum(1 for r in self.results if r.passed)
        n_fail = len(self.results) - n_pass
        print(f'=== Data Contract Validation: {self.contract.name} v{self.contract.version} ===')
        print(f'Checks: {len(self.results)} total, {n_pass} passed, {n_fail} failed')
        print()
        for r in self.results:
            status = 'PASS' if r.passed else r.severity.upper()
            print(f'  [{status:7s}] {r.check_name:20s} | {r.column:15s} | {r.message}')

print('Data contract framework implemented.')

In [ ]:
# ── Define contract and test data ──────────────────────────────────────────
np.random.seed(42)

contract = DataContract(name='training_dataset', version='1.0.0', expected_n_rows=1000)
contract.add_column(ColumnContract('feature_a', 'numeric', min_val=-5, max_val=5,
                                    expected_mean=0.0, expected_std=1.0))
contract.add_column(ColumnContract('feature_b', 'numeric', min_val=0, max_val=100,
                                    expected_mean=50.0, expected_std=15.0))
contract.add_column(ColumnContract('category', 'categorical',
                                    allowed_values=['A', 'B', 'C', 'D']))
contract.add_column(ColumnContract('target', 'numeric', min_val=0, max_val=4))

# Clean dataset
n = 1000
clean_data = {
    'feature_a': np.random.randn(n),
    'feature_b': np.random.normal(50, 15, n).clip(0, 100),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n),
    'target': np.random.randint(0, 5, n).astype(float),
}

# Drifted dataset (simulating production failure)
drifted_data = {
    'feature_a': np.random.randn(980) + 2.0,  # Mean shifted!
    'feature_b': np.random.normal(70, 15, 980).clip(0, 100),  # Mean drifted
    'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], 980),  # Unknown category!
    'target': np.random.randint(0, 5, 980).astype(float),
}

print('=== Clean Dataset ===' )
validator_clean = DataContractValidator(contract)
passed_clean = validator_clean.validate(clean_data)
validator_clean.report()
print()
print('=== Drifted Dataset ===')
validator_drifted = DataContractValidator(contract)
passed_drifted = validator_drifted.validate(drifted_data)
validator_drifted.report()
print()
print(f'CI gate result: clean={"PASS" if passed_clean else "FAIL"}, '
      f'drifted={"PASS" if passed_drifted else "FAIL"}')